# Notebook 06 — End-to-End Flow

## สถานการณ์ตัวอย่าง

ผู้ใช้ส่งข้อความ: **"ที่จอดรถโรงพยาบาล A อยู่ที่ไหน"**

เราจะ trace การทำงานทั้งหมดตั้งแต่ LINE webhook จนถึงคำตอบ

---

## Complete Flow Diagram

```
User: "ที่จอดรถโรงพยาบาล A อยู่ที่ไหน"
                    │
    [LINE Platform] │ HTTP POST
                    ▼
POST /webhook-dev  ──────────────────────────────── [FastAPI]
  │
  ├─ asyncio.create_task()  ← return 200 OK ทันที
  │
  └─► _process_one_event()
        │
        ├─ semaphore.acquire()    ← รอถ้าไม่ว่าง
        ├─ extract_line_event()   ← normalize event dict
        ├─ show_loading_animation() ← ผู้ใช้เห็นจุด 3 จุด
        │
        └─► handle_line_event()  ──────────────── [LINE Handler]
              │
              ├─ load_session("U123")  ← memory/redis
              │   → Session{state: {task_status: COMPLETED}, messages: []}
              │
              ├─ prepare messages  ← clear (COMPLETED) + add HumanMessage
              │
              └─► run_ai_runtime()  ─────────────── [LangGraph]
                    │
                    │ ainvoke(state_in)
                    │
                    ├─► session_gate  ← last_user_message = "ที่จอดรถ..."
                    │       ↓ router(COMPLETED) → "intent_classify"
                    │
                    ├─► intent_classify
                    │       ├─ LLM: user_goal = "ต้องการข้อมูลที่จอดรถ รพ.A"
                    │       └─ LLM: route = "hospital_info_agent_module"
                    │
                    └─► hospital_info_agent_module (subgraph)
                            │
                            ├─► route_hospital_info → "extract_info_tool"
                            ├─► extract_info_tool
                            │       ├─ LLM: topic_intents = ["parking_info"]
                            │       └─ LLM: hospitals = ["hospital_a"]
                            │
                            ├─► route_hospital_info → "fetch_data_tool"
                            ├─► fetch_data_tool
                            │       └─ CSV lookup: hospital_a → {parking_info: "..."}
                            │
                            ├─► route_hospital_info → "build_answer_tool"
                            └─► build_answer_tool
                                    └─ LLM: final_answer = "โรงพยาบาล A มีที่จอดรถ..."
                    │
                    └─ return state_out
              │
              ├─ save_ai_response() ← เพิ่ม AIMessage
              ├─ save_session()     ← บันทึก state
              └─ return HandlerResult
        │
        └─► reply_to_line(reply_token, final_answer)
              │
              └─ HTTP POST → LINE API
                    │
                    ▼
User: ✓ เห็นคำตอบ
```


## ทำ End-to-End Simulation

In [ ]:
import asyncio
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional

# ─────────────────────────────────────────────────────
# SETUP: Infrastructure stubs
# ─────────────────────────────────────────────────────

@dataclass
class Session:
    user_id: str
    state: Dict = field(default_factory=dict)
    messages: List = field(default_factory=list)
    display_name: Optional[str] = None

_SESSION_STORE = {}

async def load_session(user_id):
    s = _SESSION_STORE.get(user_id)
    print(f"  [session_store] LOAD {user_id} → {'found' if s else 'new session'}")
    return s

async def save_session(session):
    _SESSION_STORE[session.user_id] = session
    print(f"  [session_store] SAVE {session.user_id}")


# ─────────────────────────────────────────────────────
# LINE Integration stubs
# ─────────────────────────────────────────────────────

def extract_line_event(raw_body):
    events = raw_body.get("events", [])
    if not events: return None
    e = events[0]
    return {
        "event_type": e['type'],
        "reply_token": e.get('replyToken'),
        "user_id": e['source']['userId'],
        "message_text": e.get('message', {}).get('text') if e['type'] == 'message' else None,
        "postback_data": e.get('postback', {}).get('data') if e['type'] == 'postback' else None,
        "__line_channel_access_token": e.get('__line_channel_access_token', ''),
        "__line_account": e.get('__line_account', 'dev'),
    }

async def show_loading_animation(user_id, token):
    print(f"  [LINE] ⏳ Showing loading animation for {user_id}")

async def reply_to_line(reply_token, text=None, messages=None, quick_replies=None, token=''):
    print(f"  [LINE] 📤 Sending reply:")
    if text:
        print(f"    text: {text[:80]}{'...' if len(text) > 80 else ''}")
    if messages:
        print(f"    flex messages: {len(messages)} items")


# ─────────────────────────────────────────────────────
# Hospital CSV Data (mock)
# ─────────────────────────────────────────────────────

HOSPITAL_INDEX = {
    "hospital_a": {
        "hospital_name_th": "โรงพยาบาล A",
        "parking_info": "มีที่จอดรถ 500 คัน ชั้น B1-B3 ค่าบริการ 20 บาท/ชั่วโมง",
        "operating_hours": "เปิด 24 ชั่วโมง",
    },
    "hospital_b": {
        "hospital_name_th": "โรงพยาบาล B",
        "parking_info": "ที่จอดรถ 300 คัน ชั้น 1-2",
        "operating_hours": "จันทร์-เสาร์ 08:00-20:00",
    },
}

print("Infrastructure ready.")
print(f"Hospital index: {list(HOSPITAL_INDEX.keys())}")

In [ ]:
# ─────────────────────────────────────────────────────
# LANGGRAPH: Mock implementation
# ─────────────────────────────────────────────────────

async def mock_langgraph_invoke(state_in: dict) -> dict:
    """
    จำลองการทำงานของ LangGraph
    Trace แต่ละ node
    """
    state = dict(state_in)
    user_msg = state.get("latest_user_input", "")
    postback = state.get("postback_data", "")
    
    print()
    print("  ┌─── LangGraph Execution ───")
    
    # ── Node 1: session_gate ──
    print("  │")
    print("  ├─ [session_gate]")
    state["last_user_message"] = user_msg or postback
    print(f"  │    last_user_message = {state['last_user_message']!r}")
    
    # Router
    mode = state.get("conversation_mode", "AI")
    status = state.get("task_status", "COMPLETED")
    
    if mode == "HANDOFF":
        next_node = "handoff_mode"
    elif status == "ACTIVE":
        next_node = "resume_router"
    else:
        next_node = "intent_classify"
    
    print(f"  │    route → {next_node}")
    
    # ── Node 2: intent_classify ──
    if next_node == "intent_classify":
        print("  │")
        print("  ├─ [intent_classify]")
        
        # Mock LLM: extract user goal
        keywords_hospital = ["โรงพยาบาล", "ที่จอด", "เวลา", "ประกัน", "ค่า"]
        
        if any(kw in user_msg for kw in keywords_hospital):
            user_goal = f"ต้องการทราบข้อมูล: {user_msg}"
            route = "hospital_info_agent_module"
        else:
            user_goal = f"สอบถาม: {user_msg}"
            route = "concierge_agent_module"
        
        state["user_goal"] = user_goal
        state["route_module"] = route
        state["task_status"] = "ACTIVE"
        print(f"  │    user_goal = {user_goal!r}")
        print(f"  │    route_module = {route!r}")
        next_node = route
    
    # ── Module: hospital_info ──
    if next_node == "hospital_info_agent_module":
        print("  │")
        print("  ├─ [hospital_info_agent_module] (subgraph)")
        
        # Tool 1: extract_info_tool
        print("  │    ├─ extract_info_tool")
        topic = "parking_info"  # mock LLM extraction
        hospital_id = "hospital_a"  # mock extraction
        state["topic_intents"] = [topic]
        state["hospitals"] = [hospital_id]
        print(f"  │    │    topics = {[topic]}")
        print(f"  │    │    hospitals = {[hospital_id]}")
        
        # Tool 2: fetch_data_tool
        print("  │    ├─ fetch_data_tool")
        hospital_data = HOSPITAL_INDEX.get(hospital_id, {})
        result = {
            "hospital_name": hospital_data.get("hospital_name_th"),
            "data": {t: hospital_data.get(t) for t in [topic]}
        }
        state["hospital_results"] = [result]
        print(f"  │    │    fetched: {result}")
        
        # Tool 3: build_answer_tool
        print("  │    └─ build_answer_tool")
        # Mock LLM answer generation
        parking_info = result['data'].get('parking_info', 'ไม่มีข้อมูล')
        answer = f"ข้อมูลที่จอดรถ {result['hospital_name']}:\n{parking_info}"
        state["final_answer"] = answer
        state["task_status"] = "COMPLETED"
        print(f"  │         final_answer = {answer!r}")
    
    # ── concierge fallback ──
    elif next_node == "concierge_agent_module":
        print("  │")
        print("  ├─ [concierge_agent_module]")
        state["final_answer"] = "สวัสดีครับ! ผมเป็น AI Assistant ยินดีช่วยเหลือ"
        state["task_status"] = "COMPLETED"
        print(f"  │    final_answer = {state['final_answer']!r}")
    
    print("  │")
    print("  └─── END ───")
    
    return state

print("LangGraph mock ready.")

In [ ]:
# ─────────────────────────────────────────────────────
# FULL END-TO-END SIMULATION
# ─────────────────────────────────────────────────────

# Simulate LINE Webhook Raw Body
raw_webhook_body = {
    "destination": "Uxxxxx",
    "events": [
        {
            "type": "message",
            "message": {"type": "text", "text": "ที่จอดรถโรงพยาบาล A อยู่ที่ไหน"},
            "source": {"type": "user", "userId": "U1234567890"},
            "replyToken": "reply_token_abc123",
            "timestamp": 1713000000000,
            "mode": "active",
            "__line_channel_access_token": "dev_token_xxx",
            "__line_account": "dev",
        }
    ]
}


async def run_full_simulation():
    print("=" * 60)
    print("END-TO-END SIMULATION")
    print("=" * 60)
    print()
    
    # ═══ Step 1: FastAPI receives webhook ═══
    print("[STEP 1] FastAPI receives webhook")
    event = extract_line_event(raw_webhook_body)
    user_id = event['user_id']
    reply_token = event['reply_token']
    token = event['__line_channel_access_token']
    print(f"  user_id: {user_id}")
    print(f"  message: {event['message_text']!r}")
    print("  → return 200 OK to LINE (async background task)")
    print()
    
    # ═══ Step 2: Show loading animation ═══
    print("[STEP 2] Show loading animation")
    await show_loading_animation(user_id, token)
    print()
    
    # ═══ Step 3: Load Session ═══
    print("[STEP 3] Load session")
    session = await load_session(user_id)
    if session is None:
        session = Session(
            user_id=user_id,
            display_name="สมชาย",
            state={"task_status": "COMPLETED", "conversation_mode": "AI"}
        )
        print(f"  Created new session for {user_id}")
    print()
    
    # ═══ Step 4: Prepare State ═══
    print("[STEP 4] Prepare state for LangGraph")
    # Clear messages if new task
    if session.state.get('task_status') == 'COMPLETED':
        session.messages = []
        print("  Cleared messages (new task)")
    
    state_in = {
        **session.state,
        "messages": session.messages,
        "line_user_id": user_id,
        "line_account": event['__line_account'],
        "line_channel_access_token": token,
        "latest_user_input": event['message_text'] or "",
        "postback_data": event['postback_data'] or "",
    }
    print(f"  state_in keys: {list(state_in.keys())}")
    print()
    
    # ═══ Step 5: LangGraph Invoke ═══
    print("[STEP 5] LangGraph Invoke")
    state_out = await mock_langgraph_invoke(state_in)
    print()
    
    # ═══ Step 6: Save Session ═══
    print("[STEP 6] Save session")
    session.state = {
        k: v for k, v in state_out.items()
        if k not in ['messages', 'line_payload', 'line_messages']
    }
    final_answer = state_out.get('final_answer', '')
    if final_answer:
        from types import SimpleNamespace
        ai_msg = SimpleNamespace(type='ai', content=final_answer)
        session.messages.append(ai_msg)
    await save_session(session)
    print()
    
    # ═══ Step 7: Reply to LINE ═══
    print("[STEP 7] Reply to LINE")
    await reply_to_line(
        reply_token=reply_token,
        text=state_out.get('final_answer'),
        messages=state_out.get('line_messages', []),
        quick_replies=state_out.get('quick_replies', []),
        token=token
    )
    
    print()
    print("=" * 60)
    print("✓ Request completed!")
    print("=" * 60)
    print()
    print("Final state saved to session:")
    print(f"  task_status: {session.state.get('task_status')}")
    print(f"  messages in history: {len(session.messages)}")


asyncio.run(run_full_simulation())

## Scenario 2: Postback Event (กดปุ่ม)

ทดสอบสถานการณ์เมื่อผู้ใช้กดปุ่ม Rich Menu

In [ ]:
# Postback event
postback_body = {
    "events": [
        {
            "type": "postback",
            "postback": {"data": "action=concierge_menu"},
            "source": {"type": "user", "userId": "U1234567890"},
            "replyToken": "reply_postback_456",
            "timestamp": 1713000001000,
            "mode": "active",
            "__line_channel_access_token": "dev_token_xxx",
            "__line_account": "dev",
        }
    ]
}

event = extract_line_event(postback_body)
print("Postback Event:")
print(f"  event_type: {event['event_type']}")
print(f"  postback_data: {event['postback_data']}")
print(f"  message_text: {event['message_text']}")
print()
print("การประมวลผล Postback:")
print("  1. postback_data = 'action=concierge_menu'")
print("  2. intent_classify ตรวจ postback → fast-track")
print("  3. route ไปที่ concierge_agent_module ทันที (ไม่ต้อง LLM classify)")
print("  4. concierge module สร้าง Flex Message เมนูหลัก")

## Scenario 3: Resume Active Task

In [ ]:
# สถานการณ์: task กำลัง ACTIVE (bot ถามผู้ใช้ว่าต้องการโรงพยาบาลไหน)
# ผู้ใช้ตอบกลับมา

print("Scenario: Resume Active Task")
print()
print("Request ก่อนหน้า:")
print("  Bot: 'คุณต้องการข้อมูลของโรงพยาบาลไหน? (A/B/C)'")
print("  Session saved: task_status=ACTIVE, route_module=hospital_info")
print()
print("Request ปัจจุบัน:")
print("  User: 'โรงพยาบาล B'")
print()

# session ที่ load มา
active_session_state = {
    "task_status": "ACTIVE",
    "conversation_mode": "AI",
    "route_module": "hospital_info_agent_module",
    "current_agent_module": "hospital_info_agent_module",
    "user_goal": "ต้องการข้อมูลโรงพยาบาล",
    "topic_intents": ["parking_info"],
    "hospitals": [],  # ยังไม่ได้เลือก
    "need_ask_more_info": True,
}

def simulate_session_gate_router(state):
    mode = state.get('conversation_mode', 'AI')
    status = state.get('task_status', 'COMPLETED')
    if mode == 'HANDOFF': return 'handoff_mode'
    if status == 'ACTIVE': return 'resume_router'
    return 'intent_classify'

route = simulate_session_gate_router(active_session_state)
print(f"session_gate router → {route}")
print()
print("resume_router จะ:")
print("  - ดู current_agent_module = 'hospital_info_agent_module'")
print("  - Route ไปที่ hospital_info_agent_module ทันที")
print("  - hospital_info subgraph รับ 'โรงพยาบาล B' เป็น answer ให้ hospitals=[]")
print("  - router_hospital_info → fetch_data_tool → build_answer_tool")

## สรุปทุก Scenario

| Scenario | session_gate → | Module | ผลลัพธ์ |
|---------|---------------|--------|--------|
| ข้อความใหม่ (COMPLETED) | intent_classify | hospital_info/nurse/etc. | คำตอบ |
| Postback (COMPLETED) | intent_classify | fast-track ตาม action | Flex/คำตอบ |
| ต่อ task (ACTIVE) | resume_router | ตาม current_agent_module | คำตอบ/ถามต่อ |
| HANDOFF mode | handoff_mode | - | block หรือ switch to AI |


## Key Points สรุป

```
1. FastAPI เป็น entry point รับ webhook จาก LINE
   - return 200 OK ทันที (ไม่รอ AI)
   - process ใน background async task
   - Semaphore ป้องกัน race condition

2. LINE Integration Layer
   - extract_line_event() normalize JSON
   - show_loading_animation() UX ที่ดี
   - reply_to_line() ส่งคำตอบกลับ
   - Postback vs Text handling

3. Session Store
   - Memory (dev) หรือ Redis (prod)
   - เก็บ state + messages ของแต่ละ user
   - Clear messages เมื่อเริ่ม task ใหม่

4. LangGraph
   - State TypedDict ส่งผ่าน nodes
   - session_gate → routing → module
   - Subgraphs สำหรับแต่ละ module
   - LLM ทำงานภายใน nodes (intent, extract, answer)

5. Task Lifecycle
   COMPLETED → intent_classify → ACTIVE
   ACTIVE → resume_router → (same module)
   (module finishes) → COMPLETED
```

---

## ขั้นตอนถัดไปในการพัฒนา

1. **เพิ่ม module ใหม่**: สร้าง subgraph ใหม่ + เพิ่ม route ใน intent_classify
2. **ปรับ LLM prompt**: แก้ไขใน config.py ของแต่ละ module
3. **เพิ่ม data source**: แก้ไข fetch_data_tool ใน hospital module
4. **Deploy**: ใช้ Docker + Redis + ngrok/cloud deployment